### 05. Content-Based Recommendation Engine

**Objective:** Build a reusable function `recommend_talks(title, top_n=5)` that returns the top-N most similar TED Talks based on TF-IDF and cosine similarity.

**Input:** `../data/processed/ted_talks_cleaned.csv` and the saved artefacts from Task 4.
**Key Design:** The TF-IDF matrix and similarity matrix are loaded once (or rebuilt once if missing). The `recommend_talks()` function does **not** re-vectorize on every call, making it fast and reusable.

In [1]:
import numpy as np
import pandas as pd
import os
import difflib
import joblib
from scipy.sparse import load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)

##### Load Data and Artefacts
We load the cleaned dataset and the Task 4 artefacts. If the artefacts are missing (e.g., first run or deleted), we rebuild them **once** and save them back to disk.

In [2]:
DATA_PATH = "../data/processed/ted_talks_cleaned.csv"
MATRIX_PATH = "../data/processed/tfidf_matrix.npz"
SIM_PATH = "../data/processed/cosine_sim.npy"
VECTORIZER_PATH = "../data/processed/tfidf_vectorizer.joblib"

# Load cleaned dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Load artefacts or rebuild if missing
if os.path.exists(MATRIX_PATH) and os.path.exists(SIM_PATH) and os.path.exists(VECTORIZER_PATH):
    print("Loading saved TF-IDF artefacts...")
    tfidf_matrix = load_npz(MATRIX_PATH)
    cos_sim = np.load(SIM_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
else:
    print("Artefacts not found. Rebuilding TF-IDF and Cosine Similarity (this will take a moment)...")
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
    )
    tfidf_matrix = vectorizer.fit_transform(df["combined_text_prefixed"])
    cos_sim = cosine_similarity(tfidf_matrix, dense_output=True)
    np.fill_diagonal(cos_sim, 0.0)  # Exclude self-similarity

    # Save for future use
    from scipy.sparse import save_npz
    save_npz(MATRIX_PATH, tfidf_matrix)
    np.save(SIM_PATH, cos_sim)
    joblib.dump(vectorizer, VECTORIZER_PATH)
    print("Artefacts rebuilt and saved.")

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Cosine Similarity Matrix shape: {cos_sim.shape}")

Dataset loaded: 4005 rows, 16 columns
Loading saved TF-IDF artefacts...
TF-IDF Matrix shape: (4005, 366288)
Cosine Similarity Matrix shape: (4005, 4005)


In [14]:
df.tail(3)

,talk_id,title,speaker,speaker_occupation,description,tags,transcript,url,views,duration,published_date,event,transcript_word_count,is_short_transcript,combined_text,combined_text_prefixed
4002,62263,how policing for profit undermines your rights,Dick M. Carpenter II,law researcher,many countries have an active centuries old law that allows government agencies to take your thi...,"society, law, policy, justice system, tedx",picture yourself driving down the road tomorrow heading somewhere to buy an item you found on cr...,https://www.ted.com/talks/dick_m_carpenter_ii_how_policing_for_profit_undermines_your_rights/,464414,774,2020-04-28,TEDxMileHigh,1665,False,how policing for profit undermines your rights society law policy justice system tedx many count...,title: how policing for profit undermines your rights tags: society law policy justice system te...
4003,62784,the electrifying speeches of sojourner truth,Daina Ramey Berry,unknown,isabella baumfree was born into slavery in late 18th century new york fleeing bondage with her y...,"ted ed, education, animation, united states, women, feminism, slavery, race, history, activism, ...",in early 1828 sojourner truth approached the grand jury of kingston new york she had no experien...,https://www.ted.com/talks/daina_ramey_berry_the_electrifying_speeches_of_sojourner_truth/,56582,257,2020-04-30,TED-Ed,691,False,the electrifying speeches of sojourner truth ted ed education animation united states women femi...,title: the electrifying speeches of sojourner truth tags: ted ed education animation united stat...
4004,62794,the most important anus in the ocean,Cella Wright,unknown,is it a fuzzy sock an overripe banana a moldy tube of toothpaste in fact it s a humble sea cucum...,"animals, ted ed, animation, oceans, science, education, biology, climate change",can you guess what you re looking at is it a fuzzy sock an overripe banana a moldy tube of tooth...,https://www.ted.com/talks/cella_wright_the_most_important_anus_in_the_ocean/,0,281,2020-04-30,TED-Ed,660,False,the most important anus in the ocean animals ted ed animation oceans science education biology c...,title: the most important anus in the ocean tags: animals ted ed animation oceans science educat...


##### Build the Recommendation Function
This function accepts a TED Talk title. It handles:
- Exact (case-insensitive) matches.
- Invalid titles by suggesting the closest available title via fuzzy matching.
- A `top_n` parameter (default 5) with validation (must be between 1 and 20).
- Excludes the selected TED talk from its own recommendations.

In [ ]:
def recommend_talks(title, top_n=5):
    """
    Recommends the top-N most similar TED Talks based on a given title.

    Matching strategy (in order):
      1. Exact title match (case-insensitive, whitespace-stripped)
      2. Partial / substring match (case-insensitive)
         - If exactly one talk matches -> use it
         - If multiple talks match    -> auto-pick the closest via difflib
      3. Invalid title -> suggest the closest available title via difflib

    Parameters:
    - title (str): The title of the TED Talk (case-insensitive, exact or partial).
    - top_n (int): Number of recommendations to return (default 5). Must be 1-20.

    Returns:
    - pd.DataFrame: Ranked recommendations with Rank, Talk ID, Title, Score.
    - None: If the title cannot be resolved.
    """
    # 1. Validate top_n
    if not isinstance(top_n, int) or top_n < 1 or top_n > 20:
        print("Error: top_n must be an integer between 1 and 20.")
        return None

    # 2. Normalize input
    title_clean = title.strip().lower()
    titles_lower = df["title"].str.lower()

    # 3. Exact match first
    matches = df.index[titles_lower == title_clean].tolist()

    # 3b. Partial / substring match
    if not matches:
        partial_matches = df.index[
            titles_lower.str.contains(title_clean, regex=False, na=False)
        ].tolist()

        if len(partial_matches) == 1:
            matches = partial_matches
        elif len(partial_matches) > 1:
            # Multiple partial matches -> pick the closest by difflib
            candidate_titles = df.loc[partial_matches, "title"].tolist()
            best = difflib.get_close_matches(title, candidate_titles, n=1, cutoff=0.0)
            if best:
                best_idx = df.index[df["title"] == best[0]].tolist()[0]
                matches = [best_idx]
                print(f"Multiple matches for '{title}'. Auto-picked: '{best[0]}'")

    # 4. Still nothing -> invalid title, suggest closest
    if not matches:
        all_titles = df["title"].tolist()
        suggestions = difflib.get_close_matches(title, all_titles, n=1, cutoff=0.5)
        if suggestions:
            print(f"Invalid title: '{title}' not found.")
            print(f"Did you mean: '{suggestions[0]}'?")
        else:
            print(f"Invalid title: '{title}' not found. No close matches available.")
        return None

    # 5. Seed index
    seed_idx = matches[0]
    seed_title = df.loc[seed_idx, "title"]

    # 6. Similarity scores for this seed
    scores = cos_sim[seed_idx]

    # 7. Sort descending, exclude self
    top_indices = scores.argsort()[::-1]
    top_indices = [i for i in top_indices if i != seed_idx][:top_n]

    # 8. Build result DataFrame
    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            "Rank": rank,
            "Talk ID": int(df.loc[idx, "talk_id"]),
            "Recommended Title": df.loc[idx, "title"],
            "Similarity Score": round(float(scores[idx]), 4),
        })

    result_df = pd.DataFrame(results)
    print(f"Seed: '{seed_title}'")
    return result_df

In [27]:
# Test 1: Exact title
recommend_talks("Averting the climate crisis", top_n=5)

Seed: 'averting the climate crisis'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,243,new thinking on the climate crisis,0.1748
1,2,54715,how we can turn the tide on climate,0.1643
2,3,2784,a climate solution where all sides can win,0.1433
3,4,2441,the case for optimism on climate change,0.1423
4,5,3579,can we stop climate change by removing co2 from the air,0.1382


In [28]:
# Test 2: Exact title, different case
recommend_talks("HOW POLICING FOR PROFIT UNDERMINES YOUR RIGHTS", top_n=5)

Seed: 'how policing for profit undermines your rights'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,2406,i love being a police officer but we need reform,0.2173
1,2,2239,the hidden reason for poverty the world needs to address now,0.1719
2,3,633,the real story of mcmafia how global crime networks work,0.1589
3,4,45612,how policewomen make communities safer,0.1526
4,5,1914,why smart statistics are the key to fighting crime,0.1458


In [29]:
# Test 3: Partial title (previously failed)
recommend_talks("climate crisis", top_n=5)

Multiple matches for 'climate crisis'. Auto-picked: 'averting the climate crisis'
Seed: 'averting the climate crisis'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,243,new thinking on the climate crisis,0.1748
1,2,54715,how we can turn the tide on climate,0.1643
2,3,2784,a climate solution where all sides can win,0.1433
3,4,2441,the case for optimism on climate change,0.1423
4,5,3579,can we stop climate change by removing co2 from the air,0.1382


In [30]:
# Test 4: Partial title, different case
recommend_talks("SCHOOLS KILL CREATIVITY", top_n=5)

Seed: 'do schools kill creativity'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,1738,how to escape education s death valley,0.1710
1,2,865,bring on the learning revolution,0.1646
2,3,209,rebuilding a neighborhood with beauty dignity hope,0.1397
3,4,61930,it s ok to feel overwhelmed here s what to do next,0.1362
4,5,2182,how to run a company with almost no rules,0.1348


In [31]:
# Test 5: Ambiguous partial (auto-picks closest)
recommend_talks("simplicity", top_n=5)

Multiple matches for 'simplicity'. Auto-picked: 'simplicity sells'
Seed: 'simplicity sells'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,495,cool tricks your phone can do,0.1527
1,2,1268,6 ways to save the internet,0.1413
2,3,172,designing for simplicity,0.1348
3,4,2664,meet the inventor of the electronic spreadsheet,0.1342
4,5,845,toward a science of simplicity,0.1338


In [32]:
# Test 6: Typo / invalid title (suggests closest)
recommend_talks("Simplicity sale", top_n=5)

Invalid title: 'Simplicity sale' not found.
Did you mean: 'simplicity sells'?


In [33]:
# Test 7: Invalid top_n (validation)
recommend_talks("Averting the climate crisis", top_n=0)

Error: top_n must be an integer between 1 and 20.


In [34]:
# Test 8: Custom top_n
recommend_talks("Averting the climate crisis", top_n=10)

Seed: 'averting the climate crisis'


,Rank,Talk ID,Recommended Title,Similarity Score
0,1,243,new thinking on the climate crisis,0.1748
1,2,54715,how we can turn the tide on climate,0.1643
2,3,2784,a climate solution where all sides can win,0.1433
3,4,2441,the case for optimism on climate change,0.1423
4,5,3579,can we stop climate change by removing co2 from the air,0.1382
5,6,2339,climate change is happening here s how we adapt,0.1339
6,7,60080,how we could change the planet s climate future,0.1338
7,8,27793,100 solutions to reverse global warming,0.1316
8,9,2452,this country isn t just carbon neutral it s carbon negative,0.1304
9,10,535,what comes after an inconvenient truth,0.1303


In [36]:
# Quick sanity check: Ensure the seed talk is not in its own recommendations
seed = "Averting the climate crisis"
recs = recommend_talks(seed, top_n=5)
if recs is not None:
    seed_id = int(df[df["title"].str.lower() == seed.lower()]["talk_id"].iloc[0])
    assert seed_id not in recs["Talk ID"].values, "Self-recommendation detected!"
    print("Sanity check passed: Seed talk is excluded from recommendations.")

Seed: 'averting the climate crisis'
Sanity check passed: Seed talk is excluded from recommendations.


#### Observations and Review Questions

**Matching behaviour (updated):**
1. **Exact match** - case-insensitive, whitespace-stripped (e.g. `"  Averting the climate crisis  "` works).
2. **Partial match** - substring search (e.g. `"climate crisis"` finds `averting the climate crisis`).
   - If **one** result -> used directly.
   - If **multiple** results -> `difflib` auto-picks the closest title and prints which one was chosen.
3. **Invalid match** - `difflib.get_close_matches` suggests the closest valid title; returns `None`.
4. **Self-exclusion** - diagonal of `cos_sim` was zeroed in Task 4, and the function additionally filters out the seed index.
5. **top_n validation** - must be an integer in `[1, 20]`; default is 5.

**Review Questions:**
- **How does the function handle partial titles?** It does an exact match first; if that fails, it does a substring match. If multiple titles contain the substring, `difflib` picks the closest one automatically.
- **How does it handle invalid titles?** It suggests the closest available title via `difflib` and returns `None`.
- **Why zero the diagonal?** `cosine_similarity(talk_i, talk_i) = 1.0`. Zeroing prevents self-recommendation even if the filtering logic changes.
- **Is the function reusable?** Yes - TF-IDF and cosine similarity are computed once and saved/loaded; each call is a cheap array lookup.
- **What if `top_n` is invalid?** It returns an error message and `None`.

#### 📄 Task Report

The written report for this notebook lives at [`reports/task_05_observations.md`](../reports/task_05_observations.md).

- Markdown: [`reports/task_05_observations.md`](../reports/task_05_observations.md)
- PDF: [`reports/pdf/task_05_observations.pdf`](../reports/pdf/task_05_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
